# Breast · GSE161529

This single-cell RNA-seq notebook uses the dataset-specific marker panel in the cells below. Prepare the inputs listed in [the data guide](../../data/README.md). Start with a fresh kernel and run the annotation cells in order. The last section contains optional comparison methods. The notebook writes its files under `results/runs/single_cell/breast_gse161529/`.


In [ ]:
from pathlib import Path
import os

# Both a kernel started alongside the notebook and one started at the repository root work.
start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "data" / "datasets.tsv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from a checkout containing data/datasets.tsv.")
RAW = ROOT / "data" / "raw"
PREPARED = ROOT / "data" / "prepared"
RUN_DIR = ROOT / "results" / "runs" / "single_cell" / "breast_gse161529"
RUN_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(RUN_DIR)
RUN_COMPARISONS = False     # Enable after installing the optional comparison tools.


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

# =========================================================
# 0) Load the GSE161529 input object
# =========================================================
gse_h5ad = PREPARED / "GSE161529_combined.h5ad"
adata = sc.read_h5ad(gse_h5ad)

# Make names safe & consistent
adata.var_names = adata.var_names.astype(str)
adata.var_names_make_unique()
adata.obs_names_make_unique()

print(f"START  | cells={adata.n_obs:,}  genes={adata.n_vars:,}")
print("Obs columns:", list(adata.obs.columns)[:15], "...")
print("Var columns:", list(adata.var.columns)[:15], "...")

# Optional: show sample distribution if present
for c in ["sample", "batch"]:
    if c in adata.obs.columns:
        print(f"\n{c} counts:\n", adata.obs[c].value_counts().head(10))

# =========================================================
# 1) Define QC gene sets (mt/ribo/hb) in adata.var
# =========================================================
vn_up = adata.var_names.str.upper()

adata.var["mt"]   = vn_up.str.startswith("MT-")
adata.var["ribo"] = vn_up.str.startswith(("RPS","RPL","MRPS","MRPL"))
adata.var["hb"]   = adata.var_names.str.match(r"^(HB[ABEDM][A-Z0-9]*)", case=False)

# =========================================================
# 2) Compute QC metrics
# =========================================================
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "ribo", "hb"],
    percent_top=None,
    log1p=False,
    inplace=True
)

print("\nQC summary:")
print(adata.obs[["n_genes_by_counts","total_counts","pct_counts_mt"]].describe())

# =========================================================
# 3) Cell filtering thresholds (data-driven)
# =========================================================
min_genes = 200
min_cells_gene = 3

umi_hi   = np.percentile(adata.obs["total_counts"], 99.5)
genes_hi = np.percentile(adata.obs["n_genes_by_counts"], 99.5)
mito_hi  = max(20.0, np.percentile(adata.obs["pct_counts_mt"], 95))

print(f"\nThresholds → min_genes={min_genes}, umi_hi≈{umi_hi:.0f}, genes_hi≈{genes_hi:.0f}, mito_hi≈{mito_hi:.1f}%")

keep_cells = (
    (adata.obs["n_genes_by_counts"] >= min_genes) &
    (adata.obs["n_genes_by_counts"] <= genes_hi) &
    (adata.obs["total_counts"]      <= umi_hi) &
    (adata.obs["pct_counts_mt"]     <= mito_hi)
)

before = adata.n_obs
adata = adata[keep_cells].copy()
print(f"CELL FILTER | kept {adata.n_obs:,}/{before:,} ({adata.n_obs/before:.1%})")

# =========================================================
# 4) Gene filtering
# =========================================================
before_g = adata.n_vars
sc.pp.filter_genes(adata, min_cells=min_cells_gene)
print(f"GENE FILTER | kept {adata.n_vars:,}/{before_g:,} (≥{min_cells_gene} cells)")

# =========================================================
# 5) Normalize & log1p
# =========================================================
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata

print(f"READY | cells={adata.n_obs:,} genes={adata.n_vars:,}")



In [ ]:
import numpy as np
import scanpy as sc

#sampling

def stratified_downsample_by_sample(
    adata,
    sample_col="sample",
    target_total=90000,
    seed=0,
    min_per_sample=300
):
    rng = np.random.default_rng(seed)

    if sample_col not in adata.obs:
        raise ValueError(f"{sample_col} not found in adata.obs")

    counts = adata.obs[sample_col].value_counts()
    N = adata.n_obs

    # proportional allocation + minimum
    alloc = (counts / N * target_total).round().astype(int)
    alloc = alloc.clip(lower=min_per_sample)

    # If allocation too large, scale down
    if alloc.sum() > target_total:
        scale = target_total / alloc.sum()
        alloc = np.floor(alloc * scale).astype(int).clip(lower=1)

    # If allocation too small, fill the remainder later
    chosen = []
    all_idx_by_sample = {}

    for s, n_take in alloc.items():
        idx = np.where(adata.obs[sample_col].values == s)[0]
        all_idx_by_sample[s] = idx

        if len(idx) <= n_take:
            chosen.extend(idx.tolist())
        else:
            chosen.extend(rng.choice(idx, size=n_take, replace=False).tolist())

    chosen = np.array(chosen, dtype=int)

    # Ensure uniqueness (should already be unique, but safe)
    chosen = np.unique(chosen)

    # ---- adjust to hit exactly target_total ----
    if chosen.size > target_total:
        chosen = rng.choice(chosen, size=target_total, replace=False)

    elif chosen.size < target_total:
        need = target_total - chosen.size
        remaining = np.setdiff1d(np.arange(adata.n_obs), chosen, assume_unique=False)
        if remaining.size < need:
            raise ValueError("Not enough remaining cells to fill target_total.")
        extra = rng.choice(remaining, size=need, replace=False)
        chosen = np.concatenate([chosen, extra])

    # Shuffle indices (optional)
    rng.shuffle(chosen)

    adata_ds = adata[chosen].copy()
    print(f"Downsampled: {adata_ds.n_obs:,} cells (target {target_total:,})")
    print(adata_ds.obs[sample_col].value_counts().head(10))
    return adata_ds


# Run
adata_90k = stratified_downsample_by_sample(
    adata,
    sample_col="sample",
    target_total=90000,
    seed=42,
    min_per_sample=300
)
adata=adata_90k


In [ ]:
adata

In [ ]:
# plot density distribution


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# --- PBMC marker sets ---
PBMC_MARKERS = {
    # -----------------------------
    # Epithelial / Tumor (OSCC)
    # -----------------------------
    "Epithelial_Basal":     ["KRT5","KRT14","TP63","KRT15","ITGA6","EPCAM","KRT8","KRT18","KRT19","TACSTD2"],

    # Oral mucosa differentiation (often present in normal epithelium)
    "Keratinocyte_Diff":    ["KRT1","KRT10","IVL","LOR","FLG"],
    "Oral_Mucosa":          ["KRT4","KRT13","KRT6A","KRT16","KRT17"],

    # Proliferating tumor/epithelium
    "Cycling":              ["MKI67","TOP2A","STMN1","HMGB2","TUBA1B"],

    # EMT / invasive programs (optional but useful in tumors)
    "EMT_like":             ["VIM","TAGLN","COL1A1","FN1","SNAI2"],

    # -----------------------------
    # Fibroblast / CAF
    # -----------------------------
    "Fibroblast":           ["COL1A1","COL1A2","DCN","LUM","COL3A1"],
    "CAF_myCAF":            ["ACTA2","TAGLN","MYL9","CNN1","COL1A1"],
    "CAF_iCAF":             ["CXCL12","COL1A1","COL3A1","C7","IL6"],
    "CAF_FAP_Podoplanin":   ["FAP","PDPN","THY1","COL1A1","CXCL14"],

    # -----------------------------
    # Endothelial / Pericyte
    # -----------------------------
    "Endothelial":          ["PECAM1","VWF","KDR","EMCN","RAMP2"],
    "Lymphatic_Endo":       ["LYVE1","PDPN","PROX1","CCL21","FLT4"],
    "Pericyte":             ["RGS5","PDGFRB","CSPG4","MCAM","ABCC9"],
    "SmoothMuscle":         ["ACTA2","MYH11","TAGLN","CNN1","CALD1"],

    # -----------------------------
    # T / NK
    # -----------------------------
    "Tcell":                ["CD3D","CD3E","TRAC","LTB","IL7R"],
    "CD8_Tcell":            ["CD8A","CD8B","GZMK","NKG7","CCL5"],
    "Treg":                 ["FOXP3","IL2RA","CTLA4","IKZF2","TIGIT"],
    "NK":                   ["NKG7","GNLY","KLRD1","PRF1","GZMB"],

    # -----------------------------
    # B / Plasma
    # -----------------------------
    "Bcell":                ["MS4A1","CD79A","CD74","HLA-DRA","CD37"],
    "Plasma":               ["MZB1","JCHAIN","XBP1","SDC1","IGKC"],

    # -----------------------------
    # Myeloid / DC / Mast
    # -----------------------------
    "Myeloid_MonoMacro":    ["LYZ","LST1","TYROBP","FCER1G","CTSS"],
    "Macrophage_C1Q":       ["C1QA","C1QB","C1QC","APOE","LGALS3"],
    "Inflamm_Myeloid":      ["S100A8","S100A9","VCAN","LGALS3","IL1B"],
    "Dendritic_cDC2":       ["FCER1A","CD1C","CLEC10A","HLA-DRA","LILRA4"],
    "Dendritic_cDC1":       ["CLEC9A","XCR1","BATF3","IRF8","HLA-DRA"],
    "pDC":                  ["IL3RA","GZMB","TCF4","IRF7","SERPINF1"],
    "Mast":                 ["TPSAB1","CPA3","KIT","MS4A2","HDC"],

    # -----------------------------
    # Others sometimes present
    # -----------------------------
    "Schwann":              ["SOX10","S100B","MPZ","PLP1","PMP22"],
    "Erythroid":            ["HBB","HBA1","HBA2","ALAS2","GYPA"],
}


# --- Helper to extract vector ---
def expr_vec(adata, gene):
    x = adata[:, gene].X
    if hasattr(x, "toarray"):
        x = x.toarray()
    return np.ravel(x)

# --- Plot grouped Gaussian distributions ---
n_types = len(PBMC_MARKERS)
cols = 3
rows = int(np.ceil(n_types / cols))

fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows))
axes = axes.flatten()

for i, (ct, genes) in enumerate(PBMC_MARKERS.items()):
    ax = axes[i]
    plotted = False
    for g in genes:
        if g not in adata.var_names:
            continue
        v = expr_vec(adata, g)
        v = v[v > 0]
        if len(v) < 50:
            continue
        kde = gaussian_kde(v)
        xs = np.linspace(min(v), max(v), 400)
        ax.plot(xs, kde(xs), label=g)
        plotted = True
    if plotted:
        ax.set_title(f"{ct} markers", fontsize=12)
        ax.set_xlabel("Expression (log1p normalized)")
        ax.set_ylabel("Density")
        ax.legend(fontsize=8)
    else:
        ax.set_visible(False)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Skin data Marker Gene Distributions (Gaussian KDE per marker set)", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.97])

# --- Save the figure ---
plt.savefig("breast_marker_distributions.png", dpi=300, bbox_inches="tight")
plt.savefig("breast_marker_distributions.pdf", bbox_inches="tight")
plt.show()


In [ ]:
PBMC_MARKERS = {
# --- Breast cancer marker sets ---

    "Tumor_Epithelial": ["EPCAM","KRT8","KRT18","KRT19","MUC1"],

    "Luminal": ["ESR1","PGR","GATA3","FOXA1","KRT18"],

    "HER2_Positive": ["ERBB2","GRB7","PGAP3","STARD3","MIEN1"],

    "Basal_TNBC": ["KRT5","KRT14","KRT17","EGFR","TP63"],

    "Proliferative_Tumor": ["MKI67","TOP2A","CCNB1","AURKA","BIRC5"],


    # --- Immune cells ---

    "T_CELL": ["CD3D","CD3E","CD3G","TRAC","IL7R","FOXP3","IL2RA","CTLA4","IKZF2","TNFRSF18"],

    "B_CELL": ["CD19","MS4A1","CD79A","CD79B","BANK1"],

    "Plasma": ["MZB1","XBP1","SDC1","IGKC","JCHAIN"],

    "NK": ["NKG7","GNLY","KLRD1","KLRF1","PRF1"],


    "Dendritic": ["ITGAX","CD1C","CLEC10A","CCR7","HLA-DRA"],


    # --- Stromal cells ---

    "CAF": ["COL1A1","COL1A2","FAP","ACTA2","PDGFRA"],

    "Fibroblast": ["DCN","LUM","COL3A1","COL6A1","PDGFRB"],

    "Endothelial": ["PECAM1","VWF","KDR","CDH5","CLDN5"],

    "Pericyte": ["RGS5","PDGFRB","MCAM","KCNJ8","ABCC9"],

    "SmoothMuscle": ["ACTA2","MYH11","TAGLN","CNN1","CALD1"],

}


In [ ]:
#Method1 (Fast method without Dirichlet, marked intial supervised cell using adaptive density distribution of each marker set)
# --- PBMC marker sets ---


def adaptive_marker_threshold(adata, gene, quantile=0.75):
    vals = adata[:, gene].X
    if hasattr(vals, "toarray"):
        vals = vals.toarray()
    vals = np.asarray(vals).ravel()
    vals = vals[vals > 0]
    if vals.size == 0:
        return np.inf
    return float(np.quantile(vals, quantile))

def weak_label_quota_with_min_cells(
    adata,
    marker_sets,
    gene_quantile=0.75,
    target_frac=0.25,
    min_cells_per_type=200,
    min_score=0.2,
    score_mode="frac_high",   # "frac_high" as you used; keep for clarity
    verbose=True
):
    """
    Makes weak labels with:
      - total labeled fraction ~ target_frac
      - every labeled type has >= min_cells_per_type
      - optional min_score threshold
    """

    N = adata.n_obs
    target_total = int(round(target_frac * N))

    # --- 1) Compute per-type marker score (frac of markers above gene-specific threshold)
    scores = {}
    for cell_type, genes in marker_sets.items():
        genes = [g for g in genes if g in adata.var_names]
        if len(genes) == 0:
            continue
        thr = {g: adaptive_marker_threshold(adata, g, quantile=gene_quantile) for g in genes}

        M = sc.get.obs_df(adata, keys=list(thr.keys()))
        expr_high = M.gt(pd.Series(thr))
        frac_high = expr_high.sum(axis=1) / len(thr)

        scores[cell_type] = frac_high.astype(float)

    if len(scores) == 0:
        raise ValueError("No marker sets had genes present in adata.var_names")

    score_df = pd.DataFrame(scores, index=adata.obs_names)

    # --- 2) Best type + best score per cell
    best_score = score_df.max(axis=1).astype(float)
    best_type  = score_df.idxmax(axis=1).astype(str)

    # --- 3) Eligible cells by min_score
    eligible_mask = (best_score >= float(min_score))
    if verbose:
        print(f"[quota] Eligible (score >= {min_score}): {eligible_mask.sum()}/{N} ({eligible_mask.mean():.2%})")

    # --- 4) Keep only types that have >= min_cells_per_type eligible cells
    eligible_types_counts = best_type[eligible_mask].value_counts()
    keep_types = eligible_types_counts[eligible_types_counts >= min_cells_per_type].index.tolist()

    if verbose:
        print(f"[quota] Types with >= {min_cells_per_type} eligible cells: {len(keep_types)}/{score_df.shape[1]}")

    if len(keep_types) == 0:
        # nobody qualifies; everything unlabeled
        adata.obs["weak_label"] = pd.Categorical(["unlabeled"] * N)
        adata.obs["weak_conf"]  = best_score.values
        return adata, score_df

    # If too many types * min_cells exceeds target_total, we must drop some types
    # Keep the most abundant eligible types first.
    max_types_possible = target_total // min_cells_per_type
    if max_types_possible == 0:
        if verbose:
            print("[quota][warn] target_frac too small for min_cells_per_type. Labeling none.")
        adata.obs["weak_label"] = pd.Categorical(["unlabeled"] * N)
        adata.obs["weak_conf"]  = best_score.values
        return adata, score_df

    if len(keep_types) > max_types_possible:
        keep_types = eligible_types_counts.loc[keep_types].sort_values(ascending=False).index[:max_types_possible].tolist()
        if verbose:
            print(f"[quota][warn] Too many types for target budget; keeping top {len(keep_types)} types by eligible size.")

    # --- 5) For kept types, decide how many cells to label per type
    # Base allocation: min_cells_per_type each
    base = {t: min_cells_per_type for t in keep_types}
    base_total = sum(base.values())

    # Remaining budget to reach target_total
    remaining = max(0, target_total - base_total)

    # Available eligible cells per kept type
    avail = eligible_types_counts.loc[keep_types].to_dict()

    # Extra allocation proportional to availability beyond base
    extra = {t: 0 for t in keep_types}
    if remaining > 0:
        weights = np.array([max(0, avail[t] - base[t]) for t in keep_types], dtype=float)
        if weights.sum() > 0:
            props = weights / weights.sum()
            raw_extra = np.floor(props * remaining).astype(int)
            for t, e in zip(keep_types, raw_extra):
                extra[t] = int(e)

            # distribute leftover one-by-one
            leftover = remaining - sum(extra.values())
            if leftover > 0:
                order = np.argsort(-props)  # highest proportion first
                k = 0
                while leftover > 0:
                    t = keep_types[order[k % len(order)]]
                    extra[t] += 1
                    leftover -= 1
                    k += 1

    # Final per-type quota, capped by available eligible cells
    quota = {}
    for t in keep_types:
        q = base[t] + extra[t]
        q = min(q, avail[t])  # cannot exceed eligible availability
        quota[t] = int(q)

    # If capping reduced total, we can optionally refill from types with spare capacity
    chosen_total = sum(quota.values())
    deficit = target_total - chosen_total
    if deficit > 0:
        # refill greedily from types with spare eligible cells
        spare = {t: avail[t] - quota[t] for t in keep_types}
        order = sorted(keep_types, key=lambda t: spare[t], reverse=True)
        i = 0
        while deficit > 0 and i < len(order):
            t = order[i]
            take = min(deficit, spare[t])
            if take > 0:
                quota[t] += int(take)
                deficit -= int(take)
            i += 1

    # --- 6) Select top-scoring cells within each kept type according to quota
    weak_label = np.array(["unlabeled"] * N, dtype=object)

    # Precompute indices per type among eligible cells
    best_type_arr = best_type.values
    best_score_arr = best_score.values
    eligible_idx = np.where(eligible_mask.values)[0]

    for t in keep_types:
        idx_t = eligible_idx[best_type_arr[eligible_idx] == t]
        if idx_t.size == 0:
            continue
        # sort by best_score descending
        idx_sorted = idx_t[np.argsort(-best_score_arr[idx_t])]
        pick = idx_sorted[:quota[t]]
        weak_label[pick] = t

    adata.obs["weak_label"] = pd.Categorical(weak_label)
    adata.obs["weak_conf"]  = best_score.values

    if verbose:
        vc = adata.obs["weak_label"].value_counts()
        labeled = (adata.obs["weak_label"] != "unlabeled").sum()
        print(f"[quota] Target={target_frac:.0%} | Achieved={labeled/N:.2%} | labeled={labeled}/{N}")
        print(vc)
        small = vc[(vc.index != "unlabeled") & (vc < min_cells_per_type)]
        if len(small) > 0:
            print("[quota][ERROR] still found small labeled types:", small.to_dict())
        else:
            print(f"[quota] OK: all labeled types have >= {min_cells_per_type} cells")

    return adata, score_df


In [ ]:
#Method1
# --------------------------
# Usage (set 20–30% labeling)
# --------------------------
adata, marker_score_df = weak_label_quota_with_min_cells(
    adata,
    PBMC_MARKERS,
    gene_quantile=0.75,
    target_frac=0.6,        # 20–30% desired
    min_cells_per_type=50,  # enforce minimum cluster size
    min_score=0.2,
    verbose=True
)


In [ ]:
# Method2 with Dirichlet-process BGM + PCA
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.mixture import BayesianGaussianMixture
from sklearn.preprocessing import StandardScaler

def dp_seed_by_marker_sets_soft(
    adata,
    marker_sets,
    pca_n=10,
    n_components=30,
    weight_concentration_prior=0.05,
    random_state=0,
    per_gene_pos_quantile=0.70,
    cluster_score_min=0.20,
    min_cells_cluster=100,
    label_col_prefix="dp_seed",
    conf_col_prefix="dp_seed_conf",
    verbose=True
):
    diagnostics = {}
    all_vars = set(adata.var_names.astype(str))
    n_cells = adata.n_obs

    # --- Pre-filter thresholds ---
    min_genes_present     = 3
    min_expressed_markers = 2
    min_cells_per_gene    = 20
    min_cell_enrichment   = 0.05

    for setname, genes in marker_sets.items():
        # 0) Keep only markers present in adata
        genes = [g for g in genes if g in all_vars]
        if len(genes) < min_genes_present:
            if verbose:
                print(f"[DP-soft][skip] {setname}: only {len(genes)} markers present in adata.")
            continue

        # 1) Extract expression
        Xg = adata[:, genes].X
        if hasattr(Xg, "toarray"):
            Xg = Xg.toarray()
        Xg = np.asarray(Xg, dtype=float)

        # 2) Pre-filter 1: expressed markers
        expr_per_gene = (Xg > 0).sum(axis=0)
        expressed_markers = int(np.sum(expr_per_gene >= min_cells_per_gene))
        if expressed_markers < min_expressed_markers:
            if verbose:
                print(f"[DP-soft][skip] {setname}: only {expressed_markers} expressed markers.")
            continue

        # 3) Pre-filter 2: cell enrichment
        cells_with_any_marker = (Xg > 0).sum(axis=1) > 0
        cell_enrichment = float(np.mean(cells_with_any_marker))
        if cell_enrichment < min_cell_enrichment:
            if verbose:
                print(f"[DP-soft][skip] {setname}: cell enrichment {cell_enrichment:.4f} < {min_cell_enrichment}.")
            continue

        # 4) Per-gene thresholds
        gene_thr = {}
        for j, g in enumerate(genes):
            v = Xg[:, j]
            pos = v[v > 0]
            if pos.size >= 20:
                gene_thr[g] = float(np.quantile(pos, per_gene_pos_quantile))
            elif pos.size > 0:
                gene_thr[g] = float(np.median(pos))
            else:
                gene_thr[g] = np.inf

        # 5) Cell marker scores
        high_mask = np.zeros_like(Xg, dtype=bool)
        for j, g in enumerate(genes):
            thr = gene_thr[g]
            if np.isinf(thr):
                continue
            high_mask[:, j] = Xg[:, j] > thr

        marker_score = high_mask.sum(axis=1) / max(1, len(genes))

        # 6) Standardize + PCA + BGM
        scaler = StandardScaler(with_mean=True, with_std=True)
        Xg_scaled = scaler.fit_transform(Xg)

        # --- PCA added here ---
        # safe PCA dimension
        n_pcs = min(pca_n, Xg_scaled.shape[1], Xg_scaled.shape[0] - 1)

        # if marker panel is too small, PCA may be unnecessary
        if n_pcs >= 2:
            pca = PCA(n_components=n_pcs, random_state=random_state)
            Z = pca.fit_transform(Xg_scaled)
        else:
            Z = Xg_scaled

        bgm = BayesianGaussianMixture(
            n_components=n_components,
            weight_concentration_prior_type="dirichlet_process",
            weight_concentration_prior=weight_concentration_prior,
            covariance_type="full",
            max_iter=1000,
            n_init=1,
            random_state=random_state,
        )
        bgm.fit(Z)
        cl = bgm.predict(Z)

        # 7) Cluster-level stats
        df_cluster = pd.DataFrame({"cluster": cl, "score": marker_score})
        cluster_summary = df_cluster.groupby("cluster")["score"].agg(["mean", "count"])

        good_clusters = cluster_summary[
            (cluster_summary["mean"] >= cluster_score_min) &
            (cluster_summary["count"] >= min_cells_cluster)
        ].index.tolist()

        # 8) Assign labels + confidence
        label_col = f"{label_col_prefix}_{setname}"
        conf_col  = f"{conf_col_prefix}_{setname}"

        labels = np.array(["unlabeled"] * n_cells, dtype=object)
        confs  = np.zeros(n_cells, dtype=float)

        if len(good_clusters) > 0:
            max_score = marker_score.max() if marker_score.max() > 0 else 1.0
            for cid in good_clusters:
                idx = np.where(cl == cid)[0]
                labels[idx] = setname
                confs[idx] = marker_score[idx] / max_score

        adata.obs[label_col] = pd.Categorical(labels)
        adata.obs[conf_col]  = confs

        # 9) Diagnostics
        n_lab = (labels != "unlabeled").sum()
        pct_lab = n_lab / n_cells * 100.0
        diagnostics[setname] = {
            "n_markers_used": len(genes),
            "n_clusters": int(len(cluster_summary)),
            "good_clusters": good_clusters,
            "n_labelled": int(n_lab),
            "pct_labelled": float(pct_lab),
            "cluster_summary": cluster_summary,
            "expressed_markers": expressed_markers,
            "cell_enrichment": cell_enrichment,
            "n_pcs_used": int(n_pcs) if n_pcs >= 2 else Xg_scaled.shape[1],
        }

        if verbose:
            print(f"{setname}: labelled {n_lab}/{n_cells} ({pct_lab:.2f}%), good_clusters={len(good_clusters)}, n_pcs={n_pcs}")

    # --- Build consensus weak label across marker sets ---
    label_cols = [f"{label_col_prefix}_{s}" for s in marker_sets.keys() if f"{label_col_prefix}_{s}" in adata.obs]
    conf_cols  = [f"{conf_col_prefix}_{s}" for s in marker_sets.keys() if f"{conf_col_prefix}_{s}" in adata.obs]

    if len(label_cols) > 0:
        lab_df = adata.obs[label_cols].astype(str)
        conf_df = adata.obs[conf_cols].astype(float).fillna(0.0)

        dp_consensus_label = []
        dp_consensus_conf  = []

        for i in adata.obs_names:
            row_labels = lab_df.loc[i].values
            row_confs  = conf_df.loc[i].values
            row_confs = np.where(row_labels == "unlabeled", 0.0, row_confs)

            best_idx = int(np.argmax(row_confs))
            best_conf = float(row_confs[best_idx])
            best_label = row_labels[best_idx] if best_conf > 0 else "unlabeled"

            dp_consensus_label.append(best_label)
            dp_consensus_conf.append(best_conf)

        adata.obs["weak_label"] = pd.Categorical(dp_consensus_label)
        adata.obs["weak_conf"]  = np.array(dp_consensus_conf, dtype=float)

        # --- Looser PBMC-friendly final filter ---
        threshold = max(100, int(0.002 * adata.n_obs))   # 0.2% or at least 100 cells
        counts = adata.obs["weak_label"].value_counts()

        keep_types = set(counts[counts >= threshold].index.tolist())

        essential_types = {
    "Tumor_Epithelial",

    "Luminal",

    "HER2_Positive",

    "Basal_TNBC",

    "Proliferative_Tumor",

    "T_CELL",

    "B_CELL",

    "Plasma",

    "NK",


    "Dendritic",

    "CAF",

    "Fibroblast",

    "Endothelial",

    "Pericyte",

    "SmoothMuscle",

            
        }

        keep_types = keep_types.union(essential_types)

        mask_keep = adata.obs["weak_label"].isin(keep_types)
        adata.obs.loc[~mask_keep, "weak_label"] = "unlabeled"
        adata.obs.loc[~mask_keep, "weak_conf"]  = 0.0

        if verbose:
            print("\n[DP-soft] Final weak_label distribution:")
            print(adata.obs["weak_label"].value_counts(normalize=True) * 100)

    return diagnostics

In [ ]:
import numpy as np

N = adata.n_obs
n_components = min(300, int(np.sqrt(N))) # capped adaptive choice

diag = dp_seed_by_marker_sets_soft(
    adata,
    PBMC_MARKERS,
    pca_n=10,                       # PCA dimension for each marker block
    per_gene_pos_quantile=0.4,
    cluster_score_min=0.1,
    min_cells_cluster=100,
    n_components=n_components,
    weight_concentration_prior=0.05,
    random_state=42
)

for k, v in diag.items():
    print(f"{k}: labelled {v['n_labelled']} cells ({v['pct_labelled']:.2f}%)")

In [ ]:
#Method3 ----------GCN based labelling
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse


from scipy import sparse

def gcn_seed_labeling(
    adata,
    score_df,                       # pd.DataFrame: index=cells, columns=types
    target_frac=0.25,               # cap after gating (None to disable)
    min_score=0.55,                 # gate 1: top1 >= min_score
    delta=0.15,                     # gate 2: (top1 - top2) >= delta
    alpha=0.85,                     # propagation strength
    n_iter=100,                      # propagation iterations
    min_cells_per_type_pct=0.005,   # e.g., 0.005 = 0.5% of total cells
    min_cells_floor=200,            # absolute minimum
    label_col="weak_label_gcn",
    conf_col="seed_conf_gcn",
    neighbors_k=15,
    pca_comps=30,
    use_rep="X_pca",
    verbose=True
):
    """
    GCN-style label propagation on Scanpy kNN graph, then hard-label with gates:
      - top1 >= min_score
      - (top1 - top2) >= delta
      - drop types with < max(min_cells_floor, min_cells_per_type_pct*N)
      - optional cap: keep only top target_frac by confidence among gated cells

    Compatible with older AnnData versions (no .obsp_keys()).
    """

    # -------------------------
    # 0) Align score_df to adata
    # -------------------------
    if not isinstance(score_df, pd.DataFrame):
        raise ValueError("score_df must be a pandas DataFrame (cells x types).")

    score_df = score_df.copy()
    score_df.index = score_df.index.astype(str)
    score_df = score_df.reindex(adata.obs_names.astype(str))

    # drop columns that are all NaN
    score_df = score_df.loc[:, score_df.notna().any(axis=0)]
    if score_df.shape[1] == 0:
        raise ValueError("score_df has no usable type columns after filtering.")

    types = score_df.columns.astype(str).tolist()

    # fill NaN with 0
    S0 = score_df.fillna(0.0).to_numpy(dtype=float)

    # row-normalize to probabilities
    row_sum = S0.sum(axis=1, keepdims=True) + 1e-12
    Y0 = S0 / row_sum

    # -------------------------
    # 1) Ensure PCA exists
    # -------------------------
    if (use_rep not in getattr(adata, "obsm", {})) or (adata.obsm[use_rep] is None):
        sc.pp.pca(adata, n_comps=pca_comps, svd_solver="arpack")
        use_rep = "X_pca"

    # -------------------------
    # 2) Ensure neighbors graph exists
    #    Old/new Scanpy store connectivities in adata.obsp['connectivities']
    # -------------------------
    has_obsp = hasattr(adata, "obsp") and isinstance(adata.obsp, dict)
    has_conn = has_obsp and ("connectivities" in adata.obsp)

    if not has_conn:
        # build neighbors (writes into adata.obsp / adata.uns["neighbors"])
        n_pcs_use = min(pca_comps, adata.obsm[use_rep].shape[1])
        sc.pp.neighbors(adata, n_neighbors=neighbors_k, n_pcs=n_pcs_use)
        has_conn = ("connectivities" in adata.obsp)

    if not has_conn:
        # last fallback: some very old objects might store it differently
        # but in Scanpy this should normally exist after sc.pp.neighbors
        raise ValueError("Could not find neighbors connectivities in adata.obsp['connectivities'].")

    A = adata.obsp["connectivities"]
    if not sparse.isspmatrix(A):
        A = sparse.csr_matrix(A)

    # -------------------------
    # 3) Build normalized adjacency (GCN-style)
    #    A_hat = D^{-1/2} (A + I) D^{-1/2}
    # -------------------------
    N = adata.n_obs
    I = sparse.eye(N, format="csr")
    A_tilde = (A + I).tocsr()

    deg = np.asarray(A_tilde.sum(axis=1)).ravel()
    deg_inv_sqrt = 1.0 / np.sqrt(deg + 1e-12)
    D_inv_sqrt = sparse.diags(deg_inv_sqrt, format="csr")
    A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt

    # -------------------------
    # 4) Propagate
    #    Y_{t+1} = alpha A_hat Y_t + (1-alpha) Y0
    # -------------------------
    Y = Y0.copy()
    for _ in range(n_iter):
        Y = alpha * (A_hat @ Y) + (1.0 - alpha) * Y0

    if not isinstance(Y, np.ndarray):
        Y = np.asarray(Y)

    # -------------------------
    # 5) Gates: top1>=min_score and margin>=delta
    # -------------------------
    top1_idx = Y.argmax(axis=1)
    top1 = Y[np.arange(N), top1_idx]

    Y2 = Y.copy()
    Y2[np.arange(N), top1_idx] = -np.inf
    top2 = Y2.max(axis=1)

    margin = top1 - top2
    gate = (top1 >= float(min_score)) & (margin >= float(delta))

    labels = np.array(["unlabeled"] * N, dtype=object)
    labels[gate] = np.array(types, dtype=object)[top1_idx[gate]]

    conf = np.zeros(N, dtype=float)
    conf[gate] = top1[gate]

    # -------------------------
    # 6) Optional cap to target_frac
    # -------------------------
    if target_frac is not None:
        target_n = int(round(float(target_frac) * N))
        if target_n > 0:
            idx_labeled = np.where(labels != "unlabeled")[0]
            if idx_labeled.size > target_n:
                order = idx_labeled[np.argsort(-conf[idx_labeled])]
                keep = order[:target_n]
                drop = np.setdiff1d(idx_labeled, keep, assume_unique=False)
                labels[drop] = "unlabeled"
                conf[drop] = 0.0

    # -------------------------
    # 7) Drop small types (pct rule + floor)
    # -------------------------
    min_cells_pct = int(np.floor(float(min_cells_per_type_pct) * N))
    min_cells = max(int(min_cells_floor), int(min_cells_pct), 5)

    vc = pd.Series(labels).value_counts()
    keep_types = set(vc[vc >= min_cells].index.tolist())
    keep_types.discard("unlabeled")

    small_mask = (labels != "unlabeled") & (~pd.Series(labels).isin(keep_types).values)
    labels[small_mask] = "unlabeled"
    conf[small_mask] = 0.0

    # -------------------------
    # 8) Write back
    # -------------------------
    adata.obs[label_col] = pd.Categorical(labels)
    if "unlabeled" not in adata.obs[label_col].cat.categories:
        adata.obs[label_col] = adata.obs[label_col].cat.add_categories(["unlabeled"])
    adata.obs[label_col] = adata.obs[label_col].fillna("unlabeled")

    adata.obs[conf_col] = conf

    if verbose:
        print(f"[GCN] min_cells_per_type = {min_cells} (max(floor={min_cells_floor}, pct={min_cells_pct}, 5))")
        print("[GCN] final label distribution:")
        print(adata.obs[label_col].value_counts())
        print(f"[GCN] labeled fraction: {(adata.obs[label_col] != 'unlabeled').mean():.2%}")
        print(f"[GCN] gate pass (before size filter/cap): {gate.mean():.2%}")
        print(f"[GCN] median top1={np.median(top1):.3f}, median margin={np.median(margin):.3f}")

    return {
        "Y_propagated": Y,
        "top1": top1,
        "top2": top2,
        "margin": margin,
        "types": types,
        "min_cells_used": min_cells,
    }


In [ ]:
diag = gcn_seed_labeling(
    adata,
    score_df=marker_score_df,     # <-- your per-cell per-type marker score table
    target_frac=0.4,             # force 20–30% seeds (e.g., 0.20/0.25/0.30)
    min_score=0.2,
    delta=0.15,
    min_cells_per_type_pct=0.005, # 0.5% rule
    min_cells_floor=200,
    label_col="weak_label",
    conf_col="seed_conf",
    verbose=True
)


In [ ]:


# PCA
sc.pp.pca(adata, n_comps=30, svd_solver="arpack")

# Graph (only if using Leiden / GCN)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)

# Optional visualization
sc.tl.umap(adata)

In [ ]:
import numpy as np
import pandas as pd

def get_training_data(
    adata,
    X_key="X_pca",
    seed_col="weak_label",
    conf_col="weak_conf",
    min_seed_conf=0.3,
    max_pcs=30
):
    X = np.asarray(adata.obsm[X_key], dtype=np.float32)
    if X.ndim != 2:
        raise ValueError(f"{X_key} must be 2D.")

    # use only first max_pcs PCs
    X = X[:, :min(max_pcs, X.shape[1])]

    y = adata.obs[seed_col].astype(str).values

    if conf_col in adata.obs.columns:
        conf = np.asarray(adata.obs[conf_col], dtype=float)
        known_mask = (y != "unlabeled") & (conf >= min_seed_conf)
    else:
        known_mask = (y != "unlabeled")

    if known_mask.sum() < 20:
        raise ValueError(f"Too few confident labeled seeds: {known_mask.sum()}")

    return X, y, known_mask

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

def rbf_svm_labeling(
    adata,
    X_key="X_pca",
    seed_col="weak_label",
    conf_col="weak_conf",
    out_col="label_svm_rbf",
    out_conf_col="svm_rbf_conf",
    C=2.0,
    gamma="scale",
    min_seed_conf=0.3,
    min_conf=0.6,
    max_pcs=30
):
    X, y, known_mask = get_training_data(
        adata,
        X_key=X_key,
        seed_col=seed_col,
        conf_col=conf_col,
        min_seed_conf=min_seed_conf,
        max_pcs=max_pcs
    )

    clf = make_pipeline(
        StandardScaler(with_mean=True, with_std=True),
        SVC(
            C=C,
            gamma=gamma,
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42
        )
    )

    clf.fit(X[known_mask], y[known_mask])

    proba = clf.predict_proba(X)
    pred = clf.classes_[np.argmax(proba, axis=1)]
    conf = np.max(proba, axis=1)

    y_out = pred.copy()
    y_out[conf < min_conf] = "unlabeled"

    adata.obs[out_col] = pd.Categorical(y_out)
    adata.obs[out_conf_col] = conf

    return clf

clf_rbf = rbf_svm_labeling(
    adata,
    seed_col="weak_label",
    conf_col="weak_conf",
    min_seed_conf=0.30,
    min_conf=0.60,
    max_pcs=30
)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from collections import Counter

# ----- Feature space -----
X = np.asarray(adata.obsm["X_pca"][:, :30], dtype=float)

# ----- Seed labels -----
labels_seed = adata.obs["weak_label"].astype(str).values
conf = adata.obs.get("weak_conf", pd.Series(np.ones(len(labels_seed))))

# Use only confident seeds
seed_mask = (labels_seed != "unlabeled") & (conf > 0.3)

known_types = np.unique(labels_seed[seed_mask])
print("Known types:", known_types)

# ----- Fit KMeans -----
k = min(30, max(10, int(np.sqrt(adata.n_obs/2))))   # adaptive cluster number
km = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=20,
    max_iter=500
)
km.fit(X)

# ----- Compute centroids of seed types (fallback) -----
centroids = []
for t in known_types:
    centroids.append(X[seed_mask & (labels_seed == t)].mean(axis=0))
C = np.vstack(centroids)

# ----- Map clusters to labels -----
cluster_to_type = {}
cluster_conf = {}

for c in np.unique(km.labels_):

    idx = np.where(km.labels_ == c)[0]

    seed_in_c = labels_seed[idx]
    seed_in_c = seed_in_c[seed_in_c != "unlabeled"]

    # Case 1: cluster has seeds
    if len(seed_in_c) > 0:

        counts = Counter(seed_in_c)
        maj_label, maj_count = counts.most_common(1)[0]

        cluster_to_type[c] = maj_label
        cluster_conf[c] = maj_count / len(idx)

    # Case 2: no seeds → assign by nearest seed centroid
    else:

        center = km.cluster_centers_[c].reshape(1, -1)
        nearest = pairwise_distances(center, C).argmin()

        cluster_to_type[c] = known_types[nearest]
        cluster_conf[c] = 0.0


# ----- Assign labels to cells -----
labels_out = np.array([cluster_to_type[c] for c in km.labels_])
conf_out = np.array([cluster_conf[c] for c in km.labels_])

adata.obs["label_kmeans"] = pd.Categorical(labels_out)
adata.obs["kmeans_conf"] = conf_out

print("\nKMeans label distribution:")
print(adata.obs["label_kmeans"].value_counts())

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

def knn_labeling(
    adata,
    X_key="X_pca",
    seed_col="weak_label",
    conf_col="weak_conf",
    out_col="label_knn",
    out_conf_col="knn_conf",
    n_neighbors=9,
    min_seed_conf=0.3,
    min_conf=0.55,
    max_pcs=30
):
    X, y, known_mask = get_training_data(
        adata,
        X_key=X_key,
        seed_col=seed_col,
        conf_col=conf_col,
        min_seed_conf=min_seed_conf,
        max_pcs=max_pcs
    )

    knn = KNeighborsClassifier(
        n_neighbors=n_neighbors,
        weights="distance",
        p=2
    )
    knn.fit(X[known_mask], y[known_mask])

    pred = knn.predict(X)
    proba = knn.predict_proba(X)
    conf = np.max(proba, axis=1)

    y_out = pred.copy()
    y_out[conf < min_conf] = "unlabeled"

    adata.obs[out_col] = pd.Categorical(y_out)
    adata.obs[out_conf_col] = conf

    return knn

clf_knn = knn_labeling(
    adata,
    seed_col="weak_label",
    conf_col="weak_conf",
    n_neighbors=9,
    min_seed_conf=0.30,
    min_conf=0.55,
    max_pcs=30
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def rf_labeling(
    adata,
    X_key="X_pca",
    seed_col="weak_label",
    conf_col="weak_conf",
    out_col="label_rf",
    out_conf_col="rf_conf",
    n_estimators=500,
    max_depth=18,
    min_samples_leaf=5,
    min_seed_conf=0.3,
    min_conf=0.55,
    max_pcs=30,
    random_state=42
):
    X, y, known_mask = get_training_data(
        adata,
        X_key=X_key,
        seed_col=seed_col,
        conf_col=conf_col,
        min_seed_conf=min_seed_conf,
        max_pcs=max_pcs
    )

    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features="sqrt",
        n_jobs=-1,
        class_weight="balanced_subsample",
        random_state=random_state
    )

    rf.fit(X[known_mask], y[known_mask])

    pred = rf.predict(X)
    proba = rf.predict_proba(X)
    conf = np.max(proba, axis=1)

    y_out = pred.copy()
    y_out[conf < min_conf] = "unlabeled"

    adata.obs[out_col] = pd.Categorical(y_out)
    adata.obs[out_conf_col] = conf

    return rf


clf_rf = rf_labeling(
    adata,
    seed_col="weak_label",
    conf_col="weak_conf",
    min_seed_conf=0.30,
    min_conf=0.55,
    max_pcs=30
)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier

def mlp_labeling(
    adata,
    X_key="X_pca",
    seed_col="weak_label",
    conf_col="weak_conf",
    out_col="label_mlp",
    out_conf_col="mlp_conf",
    hidden_layer_sizes=(128, 64),
    alpha=1e-3,
    max_iter=400,
    min_seed_conf=0.3,
    min_conf=0.60,
    max_pcs=30,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
):
    X, y_str, known_mask = get_training_data(
        adata,
        X_key=X_key,
        seed_col=seed_col,
        conf_col=conf_col,
        min_seed_conf=min_seed_conf,
        max_pcs=max_pcs
    )

    le = LabelEncoder()
    y_known = le.fit_transform(y_str[known_mask])

    if len(le.classes_) < 2:
        raise ValueError(f"Only one labeled class present in seeds: {le.classes_[0]}")

    clf = make_pipeline(
        StandardScaler(with_mean=True, with_std=True),
        MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            activation="relu",
            solver="adam",
            alpha=alpha,
            learning_rate_init=1e-3,
            max_iter=max_iter,
            early_stopping=early_stopping,
            validation_fraction=validation_fraction,
            n_iter_no_change=20,
            random_state=random_state,
            verbose=False
        )
    )

    clf.fit(X[known_mask], y_known)

    proba = clf.predict_proba(X)
    pred_int = np.argmax(proba, axis=1)
    pred_lbl = le.inverse_transform(pred_int)
    conf = np.max(proba, axis=1)

    y_out = pred_lbl.copy()
    y_out[conf < min_conf] = "unlabeled"

    adata.obs[out_col] = pd.Categorical(y_out)
    adata.obs[out_conf_col] = conf

    return clf, le


clf_mlp, le = mlp_labeling(
    adata,
    seed_col="weak_label",
    conf_col="weak_conf",
    hidden_layer_sizes=(128, 64),
    alpha=1e-3,
    min_seed_conf=0.30,
    min_conf=0.60,
    max_pcs=30,
    random_state=42
)

In [ ]:
# 6) Consensus label for previously unknown cells
methods = np.vstack([
    adata.obs["label_svm_rbf"].astype(str).values,
    adata.obs["label_kmeans"].astype(str).values,
    adata.obs["label_knn"].astype(str).values,
    adata.obs["label_rf"].astype(str).values,
     adata.obs["label_mlp"].astype(str).values
])  # shape: 5 x n_cells

consensus = np.array(["unlabeled"] * adata.n_obs, dtype=object)
agree_frac = np.zeros(adata.n_obs)

for i in range(adata.n_obs):
    votes = [m[i] for m in methods]
    # don’t count 'unlabeled' as a valid vote if other labels exist
    filtered = [v for v in votes if v != "unlabeled"]
    if len(filtered) == 0:
        consensus[i] = "unlabeled"
        agree_frac[i] = 0.0
    else:
        c = Counter(filtered)
        maj_label, maj_votes = c.most_common(1)[0]
        consensus[i] = maj_label
        agree_frac[i] = maj_votes / 5.0

adata.obs["label_consensus"] = pd.Categorical(consensus)
adata.obs["consensus_agree_frac"] = agree_frac



In [ ]:
adata.obs["label_consensus"]

In [ ]:
import scanpy as sc

# Run Leiden clustering
sc.tl.leiden(adata, resolution=1.6, key_added="label_leiden")

# Check cluster labels
print(adata.obs["label_leiden"].value_counts())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

def clustering_metrics(adata, cluster_col, X_key="X_pca"):

    X = np.asarray(adata.obsm[X_key])
    labels = adata.obs[cluster_col].astype(str).values

    if len(np.unique(labels)) < 2:
        return {
            "method": cluster_col,
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan
        }

    return {
        "method": cluster_col,
        "silhouette": silhouette_score(X, labels),
        "calinski_harabasz": calinski_harabasz_score(X, labels),
        "davies_bouldin": davies_bouldin_score(X, labels)
    }


methods = [
    "label_leiden",               # Leiden clustering
    "label_consensus"     # your proposed method
]

rows = []

for m in methods:
    if m in adata.obs:
        rows.append(clustering_metrics(adata, m))

df_cluster_metrics = pd.DataFrame(rows)

print(df_cluster_metrics)

df_cluster_metrics.to_csv("pbmc_leiden_vs_proposed_metrics.csv", index=False)

print("Saved: pbmc_leiden_vs_proposed_metrics.csv")

In [ ]:
if RUN_COMPARISONS:
    ##Comparison code
    pass


In [ ]:
if RUN_COMPARISONS:
    import numpy as np
    import pandas as pd
    import scvi

    adata_scanvi = adata.copy()

    # create scANVI training labels
    min_conf = 0.30
    weak = adata_scanvi.obs["weak_label"].astype(str).values
    conf = np.asarray(adata_scanvi.obs["weak_conf"], dtype=float)

    use_seed = (weak != "unlabeled") & (conf >= min_conf)
    adata_scanvi.obs["label_for_scanvi"] = np.where(use_seed, weak, "unlabeled")
    adata_scanvi.obs["label_for_scanvi"] = pd.Categorical(adata_scanvi.obs["label_for_scanvi"])

    # make sure raw counts are available
    adata_scanvi.layers["counts"] = adata_scanvi.X.copy()

    adata_scanvi.var_names_make_unique()
    adata_scanvi.obs_names_make_unique()

    scvi.model.SCVI.setup_anndata(
        adata_scanvi,
        layer="counts",
        labels_key="label_for_scanvi",
    )

    scvi_model = scvi.model.SCVI(
        adata_scanvi,
        n_latent=30,
        gene_likelihood="nb",
    )
    scvi_model.train(max_epochs=15)

In [ ]:
if RUN_COMPARISONS:
    scanvi_model = scvi.model.SCANVI.from_scvi_model(
        scvi_model,
        labels_key="label_for_scanvi",
        unlabeled_category="unlabeled",
    )
    scanvi_model.train(max_epochs=50)

    adata_scanvi.obs["label_scanvi"] = scanvi_model.predict(adata_scanvi)
    adata_scanvi.obs["label_scanvi"]

In [ ]:
if RUN_COMPARISONS:
    import scanpy as sc

    sc.pp.normalize_total(adata_scanvi, target_sum=1e4)
    sc.pp.log1p(adata_scanvi)
    sc.pp.highly_variable_genes(adata_scanvi, n_top_genes=2000, subset=True)
    sc.pp.scale(adata_scanvi, max_value=10)
    sc.tl.pca(adata_scanvi, svd_solver="arpack")

In [ ]:
if RUN_COMPARISONS:
    import numpy as np
    import pandas as pd
    from sklearn.metrics import (
        silhouette_score,
        calinski_harabasz_score,
        davies_bouldin_score
    )

    def clustering_metrics(adata, cluster_col, X_key="X_pca"):
        X = np.asarray(adata.obsm[X_key])
        labels = adata.obs[cluster_col].astype(str).values

        if len(np.unique(labels)) < 2:
            return {
                "method": cluster_col,
                "silhouette": np.nan,
                "calinski_harabasz": np.nan,
                "davies_bouldin": np.nan
            }

        return {
            "method": cluster_col,
            "silhouette": silhouette_score(X, labels),
            "calinski_harabasz": calinski_harabasz_score(X, labels),
            "davies_bouldin": davies_bouldin_score(X, labels)
        }

    # only scANVI label
    rows = []

    if "label_scanvi" in adata_scanvi.obs:
        rows.append(clustering_metrics(adata_scanvi, "label_scanvi"))

    df_cluster_metrics = pd.DataFrame(rows)

    print(df_cluster_metrics)

    #df_cluster_metrics.to_csv("pbmc_scanvi_cluster_metrics.csv", index=False)

    #print("Saved: pbmc_scanvi_cluster_metrics.csv")

In [ ]:
if RUN_COMPARISONS:
    import numpy as np
    import pandas as pd
    import scanpy as sc
    from scipy import sparse
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import adjusted_rand_score, accuracy_score, confusion_matrix

    def _to_dense(X):
        return X.toarray() if sparse.issparse(X) else np.asarray(X)

    def compare_labels_ari_acc_spec_sens(
        adata,
        pred_col,
        ref_col,
        ignore_labels=("unlabeled", "Unknown", "Unlabeled"),
        evaluate_on="labeled"
    ):
        y_pred_all = adata.obs[pred_col].astype(str).values
        y_ref_all  = adata.obs[ref_col].astype(str).values

        ignore = set(map(str, ignore_labels))
        pred_labeled = ~np.isin(y_pred_all, list(ignore))
        ref_labeled  = ~np.isin(y_ref_all,  list(ignore))
        mask_labeled = pred_labeled & ref_labeled

        def _spec_sens_macro(yr, yp):
            classes = np.unique(np.concatenate([yr, yp]))
            specs, sens = [], []

            for cls in classes:
                yr_bin = (yr == cls).astype(int)
                yp_bin = (yp == cls).astype(int)

                cm = confusion_matrix(yr_bin, yp_bin, labels=[0, 1])
                tn, fp, fn, tp = cm.ravel()

                spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
                sen  = tp / (tp + fn) if (tp + fn) > 0 else np.nan

                specs.append(spec)
                sens.append(sen)

            return float(np.nanmean(specs)), float(np.nanmean(sens))

        yr = y_ref_all[mask_labeled]
        yp = y_pred_all[mask_labeled]

        if len(yr) == 0:
            return {
                "pred_col": pred_col,
                "ref_col": ref_col,
                "Accuracy": np.nan,
                "Specificity_macro": np.nan,
                "Sensitivity_macro": np.nan,
                "ARI": np.nan,
                "n_eval": 0,
            }

        spec_m, sens_m = _spec_sens_macro(yr, yp)

        return {
            "pred_col": pred_col,
            "ref_col": ref_col,
            "Accuracy": float(accuracy_score(yr, yp)),
            "Specificity_macro": float(spec_m),
            "Sensitivity_macro": float(sens_m),
            "ARI": float(adjusted_rand_score(yr, yp)),
            "n_eval": int(len(yr)),
        }

    def make_log_cpm1p(adata_in, target_sum=1e6):
        ad = adata_in.copy()
        X = _to_dense(ad.X).astype(np.float32)
        lib = X.sum(axis=1, keepdims=True)
        lib[lib == 0] = 1.0
        X_cpm = (X / lib) * float(target_sum)
        ad.X = np.log1p(X_cpm)
        return ad

    def make_log1p_norm(adata_in, target_sum=1e4):
        ad = adata_in.copy()
        sc.pp.normalize_total(ad, target_sum=target_sum)
        sc.pp.log1p(ad)
        return ad

##### 

In [ ]:
if RUN_COMPARISONS:
    import celltypist

    def run_celltypist_reference_query(
        adata,
        label_col,
        test_size=0.30,
        random_state=42,
        pred_col="label_celltypist",
    ):
        adata = adata.copy()
        adata.var_names_make_unique()
        adata.obs_names_make_unique()

        y = adata.obs[label_col].astype(str).values
        idx = np.arange(adata.n_obs)

        train_idx, test_idx = train_test_split(
            idx,
            test_size=test_size,
            stratify=y,
            random_state=random_state
        )

        ref = adata[train_idx].copy()
        qry = adata[test_idx].copy()

        ref = make_log1p_norm(ref)
        qry = make_log1p_norm(qry)

        model = celltypist.train(
            ref,
            labels=ref.obs[label_col].astype(str),
            n_jobs=-1,
            feature_selection=True,
        )

        pred = celltypist.annotate(
            qry,
            model=model,
            majority_voting=True,
        )

        # robust extraction
        pred_df = pred.predicted_labels
        if "majority_voting" in pred_df.columns:
            qry.obs[pred_col] = pred_df["majority_voting"].values
        elif "predicted_labels" in pred_df.columns:
            qry.obs[pred_col] = pred_df["predicted_labels"].values
        else:
            qry.obs[pred_col] = pred_df.iloc[:, 0].values

        metrics = compare_labels_ari_acc_spec_sens(
            qry,
            pred_col=pred_col,
            ref_col=label_col
        )
        return qry, metrics

In [ ]:
if RUN_COMPARISONS:
    # CellTypist
    adata, res_pbmc_ct = run_celltypist_reference_query(
        adata,
        label_col="weak_label",
        pred_col="label_celltypist"
    )


In [ ]:
if RUN_COMPARISONS:
    import numpy as np
    import pandas as pd
    from sklearn.metrics import (
        silhouette_score,
        calinski_harabasz_score,
        davies_bouldin_score
    )

    def clustering_metrics(adata, cluster_col, X_key="X_pca"):
        X = np.asarray(adata.obsm[X_key])
        labels = adata.obs[cluster_col].astype(str).values

        if len(np.unique(labels)) < 2:
            return {
                "method": cluster_col,
                "silhouette": np.nan,
                "calinski_harabasz": np.nan,
                "davies_bouldin": np.nan
            }

        return {
            "method": cluster_col,
            "silhouette": silhouette_score(X, labels),
            "calinski_harabasz": calinski_harabasz_score(X, labels),
            "davies_bouldin": davies_bouldin_score(X, labels)
        }

    # only scANVI label
    rows = []

    if "label_celltypist" in adata.obs:
        rows.append(clustering_metrics(adata, "label_celltypist"))

    df_cluster_metrics = pd.DataFrame(rows)

    print(df_cluster_metrics)

    #df_cluster_metrics.to_csv("pbmc_scanvi_cluster_metrics.csv", index=False)

    #print("Saved: pbmc_scanvi_cluster_metrics.csv")